# Primary RQ1 constant sensitivity

Seven registered conditions evaluate ±20% changes in the primary S7 blend, adoption and skip constants on all 136 frozen questions (952 evaluations). The original notebook logic, catalogs and rewrite files are preserved. No generation API key is needed.

Select **Runtime → Change runtime type → A100 GPU**, then **Run all**. Completed score captures and the final ZIP are retained in Drive. This notebook is a launcher; the scientific runner validates its published registration before inference.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'REGISTRATION_COMMIT_PENDING'
REPO_DIR = Path('/content/ComplianceGPT_sensitivity_v1')
OUTPUT_DIR = Path('/content/drive/MyDrive/ComplianceGPT_runs/rq1_constant_sensitivity_v1')
CONSOLE_LOG = OUTPUT_DIR.parent / 'rq1_constant_sensitivity_v1_console.log'
if len(SOURCE_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in SOURCE_COMMIT):
    raise RuntimeError('Use the finalized public notebook with an immutable source commit.')
if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
    if dirty:
        raise RuntimeError('The local checkout has changes. Use a fresh runtime and retain the Drive outputs.')
else:
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip() == SOURCE_COMMIT
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('Registered scientific source:', SOURCE_COMMIT)
print('Persistent result directory:', OUTPUT_DIR)


In [ ]:
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True, check=True)
print('Allocated GPU:', gpu.stdout.strip())
print('The notebook requests A100; the runner records the actual allocation.')
requirements = REPO_DIR / 'experiments/retriever_ablation/constant_sensitivity/requirements-colab.txt'
# These optional media packages are unnecessary for the text models and can conflict with a pinned Torch installation.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)


In [ ]:
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src') + ':' + str(REPO_DIR)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_rq1_constant_sensitivity', '-v'],
               cwd=REPO_DIR, env=test_environment, check=True)
print('Offline gate, cache, statistics and reporting checks passed.')


## Execute the registered study

The first phase builds E5 indexes and captures complete candidate scores once per question. The second phase replays the same scores under all seven conditions. Logical gate-call counts are recorded; the probe itself scores candidates for every question and is not a latency benchmark.

If the session is interrupted, rerun with the same source and runtime. Completed captures are reused from Drive. Changes to model, input or runtime fingerprints are rejected. A completed valid result is verified and retained without new inference.


In [ ]:
from datetime import datetime, timezone
command = [sys.executable, '-u', 'experiments/retriever_ablation/constant_sensitivity/run_sensitivity.py',
           '--output-dir', str(OUTPUT_DIR)]
with CONSOLE_LOG.open('a', encoding='utf-8') as log:
    log.write('\nExecution attempt: ' + datetime.now(timezone.utc).isoformat() + '\n')
    log.flush()
    process = subprocess.Popen(command, cwd=REPO_DIR, env=test_environment, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    result = process.wait()
if result != 0:
    raise RuntimeError('The study did not finish. Captures and the console log are retained in Drive; review the error before resuming.')


In [ ]:
import hashlib
import json
import zipfile

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
assert summary['accepted'] and summary['evaluations'] == 952 and summary['verified_cache_captures'] == 136
assert all(summary['acceptance_checks'].values())
archive_path = OUTPUT_DIR / 'rq1_constant_sensitivity_v1.zip'
with zipfile.ZipFile(archive_path) as bundle:
    assert bundle.testzip() is None
    for line in (OUTPUT_DIR / 'SHA256SUMS').read_text().splitlines():
        expected, relative = line.split('  ', 1)
        assert hashlib.sha256(bundle.read(relative)).hexdigest() == expected, relative
expected, name = (OUTPUT_DIR / 'ARCHIVE_SHA256SUMS').read_text().strip().split('  ', 1)
assert name == archive_path.name
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == expected
print('Verified all 952 evaluations and the complete result archive.')
print('Historical-baseline comparison:', summary['historical_baseline'])
print('ZIP:', archive_path)
print((OUTPUT_DIR / 'SUMMARY.md').read_text())


In [ ]:
from google.colab import files
files.download(str(archive_path))


## Interpretation

Return `rq1_constant_sensitivity_v1.zip` for independent artifact checks and dissertation integration after the remaining experiments. Report every condition and any deterioration. The new baseline is compared with the saved historical ranks; it does not replace them. This local sensitivity study does not remove evaluation-set tuning bias, validate author labels independently or establish held-out generalization.
